<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Warum LangGraph?
</b></font> </br></p>

---

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M08-Warum-LangGraph"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Limitierungen von create_agent()
---


`create_agent()` ist ein hervorragender Einstieg – aber es stößt bei komplexen
Szenarien an seine Grenzen.

In der Praxis tritt das Schleifen-Problem häufiger auf als erwartet: Ein Agent, der nach Tool-Ergebnissen weiterfragt, statt zu schließen, verbraucht Token und Zeit — oft wegen eines unklaren Abbruchkriteriums im Prompt.


<p><font color='black' size="5">
Die 4 Hauptprobleme
</font></p>

| # | Problem | Symptom | Beispiel |
|---|---------|---------|----------|
| 1 | **Kein persistenter Zustand** | Jeder Aufruf startet neu | "Erinnere dich an Schritt 1" scheitert |
| 2 | **Kein explizites Routing** | LLM entscheidet alles | Keine Garantie, welches Tool gewählt wird |
| 3 | **Kein Human-in-the-Loop** | Kann nicht pausieren | "Bestätige vor dem Ausführen" nicht möglich |
| 4 | **Keine parallele Ausführung** | Immer sequentiell | Mehrere Datenquellen gleichzeitig abfragen |


**Kurzregel-Übersicht**

Für einfache Agenten ohne Zustand reicht `create_react_agent()` vollständig aus — LangGraph wird dort nicht benötigt. Sobald ein Agent Gedächtnis oder persistenten Zustand braucht, ist `StateGraph` mit `MemorySaver` die richtige Wahl. Ablauflogik mit Verzweigungen wird mit `add_conditional_edges()` abgebildet, menschliche Eingriffspunkte erfordern `interrupt_before` oder `interrupt_after`, und parallele Agenten werden über Subgraphen oder parallele Edges koordiniert.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Entscheidungsregel</font> </br></p>


diagram = '''
flowchart TD
    FRAGE{"Welche Komplexität\nhat der Agent?"}
    FRAGE -->|"Einfach\nkein Zustand"| CA["create_react_agent()\nausreichend"]
    FRAGE -->|"Zustand /\nGedächtnis"| LG1["LangGraph\n+ MemorySaver"]
    FRAGE -->|"Verzweigungen /\nRouting"| LG2["LangGraph\n+ conditional_edges"]
    FRAGE -->|"Human-in-\nthe-Loop"| LG3["LangGraph\n+ interrupt"]
    FRAGE -->|"Mehrere\nAgenten"| LG4["LangGraph\n+ Subgraphen"]

    style FRAGE fill:#FF9800,color:#fff
    style CA    fill:#4CAF50,color:#fff
    style LG1   fill:#2196F3,color:#fff
    style LG2   fill:#2196F3,color:#fff
    style LG3   fill:#2196F3,color:#fff
    style LG4   fill:#2196F3,color:#fff
'''
mermaid(diagram, width=1000)


<p><font color='black' size="5">
Demonstration: Das Gedächtnis-Problem
</font></p>


Ein `create_agent()`-Agent vergisst nach jedem `.invoke()`-Aufruf alles,
was im vorherigen Aufruf passiert ist – es sei denn, die History wird manuell übergeben.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent

# Ein Zaehler-Tool
_zaehler = {"wert": 0}

@tool
def zaehler_erhoehen() -> str:
    """Erhoeht den internen Zaehler um 1 und gibt den aktuellen Wert zurueck."""
    _zaehler["wert"] += 1
    return f"Zaehler = {_zaehler['wert']}"

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)
system_prompt = (
    "Du bist ein Assistent fuer den Zaehler-Test. "
    "Fuehre Tool-Aufrufe korrekt aus und antworte knapp auf Deutsch."
)
agent = create_agent(model=llm, tools=[zaehler_erhoehen], system_prompt=system_prompt, debug=False)

# Aufruf 1
_zaehler["wert"] = 0
r1 = agent.invoke({"messages": [{"role": "user", "content": "Erhoehe den Zaehler zweimal."}]})
print(f"Nach Aufruf 1: {r1['messages'][-1].content[:120]}")

# Aufruf 2 - neuer Gespraechs-Kontext, kein Gedaechtnis
r2 = agent.invoke({"messages": [{"role": "user", "content": "Wie hoch ist der Zaehler jetzt?"}]})
print(f"Nach Aufruf 2: {r2['messages'][-1].content[:120]}")

print()
print(f"Interner Zaehler: {_zaehler['wert']}")
print("Problem: create_agent() hat keine Session-Continuity zwischen separaten .invoke()-Aufrufen.")


In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart TB
    subgraph create_agent["<b>create_agent() – Einschränkungen</b>"]
        direction LR
        U1["Aufruf 1"] --> A1["Agent"]
        A1 --> T1["Tool"]
        T1 --> A1
        A1 --> R1["Antwort"]
        R1 --> X1((" "))

        U2["Aufruf 2"] --> A2["Agent\n(neu gestartet)"]
        A2 --> R2["Antwort\n(ohne Kontext)"]

        X1 -. "Kontext verloren!" .-> A2
    end

    subgraph langgraph["<b>LangGraph – Persistenter State</b>"]
        direction TB
        V1["Aufruf 1"] --> N1["Node A"]
        N1 --> S[("Checkpoint / State\n(history, variables)")]

        V2["Aufruf 2"] --> N2["Node B"]
        S -. "lädt Kontext" .-> N2
        N2 --> S
    end

    style X1 fill:#FF6B6B
    style S fill:#90EE90
'''

mermaid(diagram, width=1200)

# 2 | State Machine Konzept
---



LangGraph basiert auf dem Konzept der **State Machine** (Zustandsautomat).  
Ein Zustandsautomat beschreibt ein System durch:

| Begriff | Erklärung | LangGraph-Entsprechung |
|---------|-----------|----------------------|
| **State** | Der aktuelle Zustand des Systems | `TypedDict` mit allen Daten |
| **Node** | Eine Aktion, die den Zustand verändert | Python-Funktion |
| **Edge** | Übergang von einem Node zum nächsten | `add_edge()` |
| **Conditional Edge** | Bedingter Übergang | `add_conditional_edges()` |
| **START** | Einstiegspunkt des Graphen | `START` Konstante |
| **END** | Ausstiegspunkt des Graphen | `END` Konstante |



**Der State als einzige Wahrheitsquelle**

Alle Nodes lesen vom State und schreiben in den State.  
Nodes kommunizieren **nicht direkt** miteinander – nur über den gemeinsamen State.

```
State = {
    "messages": [...],    # Gesprächsverlauf
    "schritt":  2,        # Aktueller Schritt
    "ergebnis": "...",    # Zwischen-/Endergebnis
}
```

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

# ─── State-Definition ────────────────────────────────────────────────────
class AgentState(TypedDict):
    """State des Agenten – wird von allen Nodes geteilt und verändert."""
    messages: Annotated[list, add_messages]  # add_messages = Nachrichten akkumulieren
    schritt:  int                            # Zählt Ausführungsschritte
    fertig:   bool                           # Signal zum Beenden

# add_messages ist ein Reducer:
# Statt list zu überschreiben, werden neue Nachrichten ANGEHÄNGT
print("State-Typen:")
print(f"  messages → list (akkumuliert via add_messages)")
print(f"  schritt  → int  (überschrieben)")
print(f"  fertig   → bool (überschrieben)")

# Initialer State
start_state: AgentState = {
    "messages": [],
    "schritt":  0,
    "fertig":   False,
}
print(f"\nStart-State: {start_state}")

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart TD
    ST([START]) --> N1

    N1["Node: llm_aufrufen()\n• liest: messages\n• schreibt: messages, schritt"] --> COND

    COND{"Bedingung:\nfertig == True?"}
    COND -->|"Nein"| N2
    COND -->|"Ja"| FIN

    N2["Node: tool_ausfuehren()\n• liest: messages\n• schreibt: messages"] --> N1

    FIN([FINISH])

    STATE[/"State (geteilt):\nmessages: [...]\nschritt: 2\nfertig: False"/]

    N1 <-.->|lesen/schreiben| STATE
    N2 <-.->|lesen/schreiben| STATE

    style ST fill:#90EE90
    style FIN fill:#FFB6C1
    style COND fill:#FFD700
    style STATE fill:#E0E0FF,stroke:#9999CC
'''

mermaid(diagram, width=450)

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

# ─── Node-Funktionen ─────────────────────────────────────────────────────
def llm_node(state: AgentState) -> dict:
    """Ruft das LLM mit dem aktuellen Nachrichtenstand auf."""
    response = llm.invoke(state["messages"])
    return {
        "messages": [response],        # Neue Nachricht wird ANGEHÄNGT
        "schritt":  state["schritt"] + 1,
        "fertig":   True               # Einfaches Beispiel: immer fertig
    }

# ─── Graph aufbauen ──────────────────────────────────────────────────────
builder = StateGraph(AgentState)

builder.add_node("llm", llm_node)  # Node registrieren
builder.add_edge(START, "llm")     # START → llm
builder.add_edge("llm", END)       # llm → END

graph = builder.compile()

print("Graph aufgebaut!")
print(f"Nodes: {list(builder.nodes.keys())}")

# 4 | Visualisierung: Graph-Diagramm
---

LangGraph kann den Graphen automatisch als Bild rendern – ideal zum Debuggen und zur Dokumentation.

```python
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))
```

> **Best Practice:** Immer nach `compile()` ausführen – zeigt den tatsächlichen,
> kompilierten Graphen (nicht nur den Builder).

> **💡 Tipp: Mermaid Live Editor**
> Eigene Graphen und Diagramme direkt im Browser entwerfen, ohne Code auszuführen:
> [https://mermaid.live](https://mermaid.live)
> Ideal zum Skizzieren der Graph-Struktur vor der Implementierung.

In [ ]:
# Graph nach compile() visualisieren (LangGraph Best Practice)
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# Ersten Graphen ausführen
initial_state: AgentState = {
    "messages": [HumanMessage(content="Was ist LangGraph in einem Satz?")],
    "schritt":  0,
    "fertig":   False,
}

ergebnis = graph.invoke(initial_state)

mprint(f"""
## Ergebnis des ersten Graphen

**Eingabe:** {initial_state["messages"][0].content}

**Antwort:** {ergebnis["messages"][-1].content}

**Schritte:** {ergebnis["schritt"]}

**Fertig:** {ergebnis["fertig"]}
""")

# 5 | LangGraph vs. LangChain
---

Beide Frameworks ergänzen sich – sie ersetzen einander **nicht**.

## Entscheidungsmatrix

| Kriterium | `create_agent()` | LangGraph |
|-----------|-----------------|----------|
| **Komplexität** | Einfach | Mittel–Hoch |
| **Zustand** | Stateless | Stateful (TypedDict) |
| **Routing** | LLM entscheidet | Explizit definiert |
| **Sessions** | Manuell | Checkpointing eingebaut |
| **Human-in-Loop** | Nicht direkt | `interrupt()` |
| **Parallele Ausführung** | Nein | Ja (Send API) |
| **Multi-Agent** | Umständlich | Erstklassig |
| **Einstieg** | ✅ Schnell | Lernkurve nötig |
| **Debugging** | LangSmith-Traces | LangSmith + Graph-Viz |

## Wann was verwenden?

```
Einfache Q&A, RAG, einzelner Agent → create_agent()

Komplexe Workflows, Persistenz, HITL, Multi-Agent → LangGraph
```

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart TD
    Q["Neue Agenten-Aufgabe"]
    Q --> A{"Zustand zwischen\nAufrufen nötig?"}

    A -->|"Nein"| B{"Routing\nvorhersehbar?"}
    A -->|"Ja"| LG

    B -->|"Egal"| C{"Multi-Agent\noder HITL?"}
    B -->|"Muss explizit\nsein"| LG

    C -->|"Nein"| CA["create_agent()"]
    C -->|"Ja"| LG["LangGraph"]

    style CA fill:#90EE90
    style LG fill:#87CEEB
    style Q fill:#FFD700
'''

mermaid(diagram, width=350)

> **📚 Weiterführend: Modell-Auswahl pro Knoten**
>
> Ab Tag 5 (*Multi-Agent Patterns*) wird relevant, **welches Modell in welchem Knoten** eingesetzt wird:
> Supervisor- und Routing-Knoten brauchen stärkere Reasoning-Modelle als Worker-Knoten.
>
> → [Modell-Auswahl Guide](https://ralf-42.github.io/Agenten/frameworks/Modell_Auswahl_Guide.html)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, es kann aber auch gerne eine andere Herausforderung angegangen werden.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Wenn bei einer Aufgabe eine Blockade entsteht, kann zum Beispiel Gemini in Google Colab verwendet werden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**

Bauen Sie einen `StateGraph` mit **genau 2 Nodes** (`analyse_node` und `antwort_node`).

- Definieren Sie einen `AgentState` mit mindestens 3 Feldern, davon eines `messages`.
- `analyse_node` liest die eingehende Nutzerfrage und schreibt eine kurze Einschätzung in den State.
- `antwort_node` liest die Einschätzung und erzeugt eine strukturierte Antwort.
- **Ziel:** Der Graph verarbeitet eine Nutzerfrage und liefert eine strukturierte Antwort.

In [ ]:
# ✅ Selbstcheck Grundlagen 🟢
# Führen Sie diesen Block aus, nachdem Sie Ihren 2-Node-Graphen gebaut haben.
# Passen Sie die Variablennamen ggf. an Ihre Implementierung an.

assert graph is not None, "graph ist None – bitte den StateGraph kompilieren."
assert 'analyse_node' in graph.nodes, "'analyse_node' nicht im Graphen – Node korrekt registriert?"
assert 'antwort_node' in graph.nodes, "'antwort_node' nicht im Graphen – Node korrekt registriert?"
print("✅ Grundlagen-Selbstcheck bestanden!")

**Aufbau**

Erweitern Sie den Graphen um einen **dritten Node** und visualisieren Sie ihn:

- Fügen Sie einen dritten Node hinzu (z. B. `zusammenfassung_node`).
- Verbinden Sie alle Nodes mit `add_edge()` zu einem sinnvollen Ablauf.
- Kompilieren Sie den Graphen und visualisieren Sie ihn mit `graph.get_graph().draw_mermaid_png()`.
- Führen Sie den Graphen mit einem Test-State aus und zeigen Sie das Ergebnis.

In [ ]:
# ✅ Selbstcheck Aufbau 🟡
# Führen Sie diesen Block aus, nachdem Sie den 3-Node-Graphen kompiliert und visualisiert haben.

assert graph is not None, "graph ist None – bitte den erweiterten Graphen kompilieren."
assert len(list(graph.nodes)) >= 3, "Weniger als 3 Nodes – dritten Node hinzufügen."
print("✅ Aufbau-Selbstcheck bestanden!")

**Vertiefung**

Bauen Sie eine **Conditional Edge** mit Routing-Logik ein:

- Fügen Sie ein Score-Feld in den State ein (z. B. `konfidenz: float`).
- Implementieren Sie eine Router-Funktion: wenn `konfidenz < 0.7` → zurück zu `analyse_node`, sonst → `END`.
- Verbinden Sie die Conditional Edge mit `add_conditional_edges()`.
- Begründen Sie kurz, warum dieser Fall mit LangGraph besser abbildbar ist als mit einem linearen Ablauf.

In [ ]:
# ✅ Selbstcheck Vertiefung 🔴
# Führen Sie diesen Block aus, nachdem Sie die Conditional Edge implementiert haben.
import os

assert graph is not None, "graph ist None – bitte den Graphen mit Conditional Edge kompilieren."
assert os.environ.get("LANGSMITH_TRACING") == "true", "LANGSMITH_TRACING ist nicht aktiv."
print("✅ Vertiefung-Selbstcheck bestanden!")